# Fine-Tuning Gemma3-1B-Instruct dengan Unsloth (Google Colab) — **v2 (perbaikan)**

> **Jalankan setelah `03_finetune_qwen35_0_8b.ipynb` dan `01_finetune_llama32_1b.ipynb`.**

Versi **v2** ini memperbaiki penyebab utama output buruk pada training sebelumnya
(val_loss ~2.85, model menjawab pertanyaan klinis Indonesia dengan ICD-11 code lookup):

1. **System prompt terpadu (root-cause fix).** Sebelumnya tiap source punya system prompt
   berbeda (9 macam) dan prompt inference (`...ICD-11 classification`) **tidak pernah ada
   di training** -> model ter-route ke gaya jawaban ICD-11 lookup. Sekarang **satu** system
   prompt dipakai konsisten di training **dan** inference, dengan instruksi menjawab dalam
   bahasa yang sama dengan pertanyaan.
2. **Completion-only training (`train_on_responses_only`, `packing=False`).** Loss kini
   dihitung **hanya pada jawaban assistant** (system+user di-mask). Ini memperbaiki val_loss
   tinggi + instruction-following lemah, sekaligus menghindari risiko packing TRL #3705.
3. **Filter ICD-11 code-lookup** (noise untuk chatbot klinis) dari train & val.
4. **Hyperparameter**: `LEARNING_RATE 2e-4 -> 1e-4`, `EPOCHS 3 -> 5`, `WARMUP_RATIO 0.05 -> 0.10`,
   + `EarlyStoppingCallback(patience=2)`.
5. QoL: verifikasi format, progress bar ROUGE-L, parameter inferensi lebih anti-repetisi.

> Data Indonesia di dataset sudah ~23% (tidak perlu upsample). Field `source` sudah dibuang
> di notebook 00, jadi inspeksi/filter di sini **berbasis konten**, bukan `source`.

**Prasyarat:** Runtime -> Change runtime type -> **GPU (T4 atau lebih baik)**.
Model `unsloth/gemma-3-1b-it-unsloth-bnb-4bit` public/ungated -> `HF_TOKEN` opsional.

In [ ]:
%%capture
!pip install unsloth
%pip install -q evaluate rouge_score matplotlib

In [ ]:
import os, gc, json
import torch
from pathlib import Path

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

assert torch.cuda.is_available(), 'CUDA tidak tersedia! Set Runtime -> Change runtime type -> GPU.'

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision('high')

print(f'GPU    : {torch.cuda.get_device_name(0)}')
print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'bf16   : {torch.cuda.is_bf16_supported()}')
print(f'PyTorch: {torch.__version__}')

## Konfigurasi

In [ ]:
DRIVE_PROJECT_DIR = '/content/drive/MyDrive/Fine-Tune SLM for Medical Chatbot'

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = Path(DRIVE_PROJECT_DIR)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    BASE_DIR = Path(os.getcwd()).parent
    if not (BASE_DIR / 'Data').exists():
        BASE_DIR = Path(os.getcwd())

assert (BASE_DIR / 'Data').exists(), f'Data/ tidak ditemukan di {BASE_DIR}'

SHARED_DIR = BASE_DIR / 'Data' / 'processed_shared'
DATA_DIR   = BASE_DIR / 'Data' / 'processed'
OUT_DIR    = BASE_DIR / 'outputs'
OUT_DIR.mkdir(parents=True, exist_ok=True)

HF_TOKEN = os.environ.get('HF_TOKEN', '')
if not HF_TOKEN and IN_COLAB:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN') or ''
    except Exception:
        pass

MODEL_ID    = 'unsloth/gemma-3-1b-it-unsloth-bnb-4bit'
# Path khusus v2 (-v2): JANGAN pakai folder checkpoint v1 -> v2 beda config
# (packing/filter/epoch) jadi tidak boleh resume dari v1, dan ini juga yang
# memicu 'KeyError: EarlyStoppingCallback' saat resume dari checkpoint v1.
# Output v1 tetap tersimpan utuh untuk pembanding skripsi.
ADAPTER_DIR = str(OUT_DIR / 'checkpoints' / 'gemma3-1b-medical-adapter-v2')
MERGED_DIR  = str(OUT_DIR / 'merged'      / 'gemma3-1b-medical-v2')
CKPT_DIR    = str(OUT_DIR / 'checkpoints' / 'gemma3-1b-train-v2')

TRAIN_RED_FILE = str(SHARED_DIR / 'train_reduced.jsonl')
VAL_RED_FILE   = str(SHARED_DIR / 'val_reduced.jsonl')
TEST_FILE      = str(DATA_DIR   / 'test.jsonl')

with open(str(SHARED_DIR / 'training_config_recommended.json')) as _f:
    _cfg = json.load(_f)
MAX_SEQ_LEN = _cfg['max_seq_length']
print(f'MAX_SEQ_LEN  : {MAX_SEQ_LEN}')
print(f'Target train : {_cfg["target_train_samples"]:,}')

# === System prompt TERPADU (dipakai sama di training & inference) ===
# Satu prompt konsisten -> hapus bug 'system-prompt routing' ke ICD-11 lookup.
# Instruksi mirror bahasa -> jawab dalam bahasa yang sama dengan pertanyaan.
UNIFIED_SYSTEM = (
    'You are a helpful and knowledgeable medical assistant. '
    'Answer the user\'s medical question accurately, clearly, and in appropriate detail, '
    'based on established clinical knowledge and guidelines. '
    'Always reply in the SAME language as the user\'s question '
    '(for example, answer in Indonesian when the question is in Indonesian). '
    'Remind the user to consult a qualified healthcare professional for personal '
    'diagnosis and treatment.'
)

LORA_R      = 16
LORA_ALPHA  = 32
LORA_DROP   = 0.05
LORA_TARGET = ['q_proj', 'k_proj', 'v_proj', 'o_proj',
               'gate_proj', 'up_proj', 'down_proj']

BATCH_SIZE    = 4
GRAD_ACCUM    = 4
LEARNING_RATE = 1e-4     # v2: 2e-4 -> 1e-4 (lebih stabil untuk 1B)
EPOCHS        = 5        # v2: 3 -> 5 (+ EarlyStopping patience=2)
WEIGHT_DECAY  = 0.01
WARMUP_RATIO  = 0.10     # v2: 0.05 -> 0.10
SEED          = 42

DO_MERGE    = True
PUSH_TO_HUB = False
HF_REPO     = 'ariesdjae/gemma3-1b-medical'

print(f'Mode     : {"Colab (Drive)" if IN_COLAB else "Lokal"}')
print(f'Model    : {MODEL_ID}')
print(f'EffBatch : {BATCH_SIZE * GRAD_ACCUM}  | LR {LEARNING_RATE} | epochs {EPOCHS}')
print(f'\nPath check:')
print(f'  SHARED_DIR     : {SHARED_DIR}  {"OK" if SHARED_DIR.exists() else "TIDAK ADA"}')
print(f'  DATA_DIR       : {DATA_DIR}  {"OK" if DATA_DIR.exists() else "TIDAK ADA"}')
print(f'  train_reduced  : {"OK" if Path(TRAIN_RED_FILE).exists() else "TIDAK ADA"}')
print(f'  val_reduced    : {"OK" if Path(VAL_RED_FILE).exists() else "TIDAK ADA"}')
print(f'  test.jsonl     : {"OK" if Path(TEST_FILE).exists() else "TIDAK ADA"}')

In [ ]:
from datasets import load_dataset

train_ds = load_dataset('json', data_files=TRAIN_RED_FILE, split='train')
val_ds   = load_dataset('json', data_files=VAL_RED_FILE,   split='train')
print(f'Train : {len(train_ds):,}  Val : {len(val_ds):,}')

## Inspeksi & Filter Dataset

Field `source` sudah dibuang di notebook 00, jadi deteksi di bawah **berbasis konten**.

In [ ]:
from collections import Counter

ID_WORDS = ['dokter', 'pasien', 'obat', 'demam', 'rumah sakit', 'gejala',
            'penyakit', 'kesehatan', 'nyeri', 'mual', 'puskesmas']

def detect_lang(sample):
    txt = ' '.join(m['content'] for m in sample['messages']).lower()
    return 'id' if sum(w in txt for w in ID_WORDS) >= 2 else 'en'

def is_icd_lookup(sample):
    # Pola ICD-11 code-lookup ('What is the ICD-11 code for X' / 'diagnosed with ICD-11 code X').
    user = next((m['content'] for m in sample['messages'] if m['role'] == 'user'), '').lower()
    return ('icd' in user and 'code' in user)

langs, icd, icd_examples = Counter(), 0, []
total = len(train_ds)
for s in train_ds:
    langs[detect_lang(s)] += 1
    if is_icd_lookup(s):
        icd += 1
        if len(icd_examples) < 3:
            icd_examples.append(next(m['content'] for m in s['messages']
                                     if m['role'] == 'user')[:140])

print(f'Total train      : {total:,}')
print(f'Perkiraan bahasa : ID={langs["id"]:,} ({langs["id"]/total*100:.1f}%)  '
      f'EN={langs["en"]:,} ({langs["en"]/total*100:.1f}%)')
print(f'ICD-11 lookup    : {icd:,} ({icd/total*100:.1f}%)  <- akan difilter')
print('\nContoh ICD-11 lookup:')
for e in icd_examples:
    print('  -', e)

In [ ]:
# Buang sampel ICD-11 code-lookup (noise untuk chatbot klinis) dari train & val.
_before_tr, _before_va = len(train_ds), len(val_ds)
train_ds = train_ds.filter(lambda s: not is_icd_lookup(s), num_proc=2)
val_ds   = val_ds.filter(lambda s: not is_icd_lookup(s),   num_proc=2)
print(f'Train : {_before_tr:,} -> {len(train_ds):,}  ({_before_tr - len(train_ds):,} dibuang)')
print(f'Val   : {_before_va:,} -> {len(val_ds):,}  ({_before_va - len(val_ds):,} dibuang)')

In [ ]:
from unsloth import FastLanguageModel

print(f'Loading {MODEL_ID} ...')
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,            # auto-detect (bf16 di T4/L4/A100)
    load_in_4bit=True,
    token=HF_TOKEN or None,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

torch.cuda.empty_cache(); gc.collect()
total = sum(p.numel() for p in model.parameters())
print(f'Params : {total / 1e9:.2f}B')
print(f'VRAM   : {torch.cuda.memory_allocated() / 1e9:.2f} GB')

In [ ]:
def to_gemma_messages(messages, system=UNIFIED_SYSTEM):
    # Gemma chat template TIDAK mendukung role 'system' (TemplateError).
    # SELALU pakai system terpadu (buang system asli per-sample) supaya konsisten
    # antara training & inference -> menghapus bug 'system-prompt routing'.
    out = [dict(m) for m in messages if m['role'] != 'system']
    if system and out and out[0]['role'] == 'user':
        out[0] = {'role': 'user', 'content': f'{system}\n\n{out[0]["content"]}'}
    return out


def format_chat(sample):
    return {'text': tokenizer.apply_chat_template(
        to_gemma_messages(sample['messages']), tokenize=False, add_generation_prompt=False
    )}

train_ds = train_ds.map(format_chat, remove_columns=train_ds.column_names, num_proc=2)
val_ds   = val_ds.map(format_chat,   remove_columns=val_ds.column_names,   num_proc=2)
print(train_ds[0]['text'][:500])

## Verifikasi format chat

In [ ]:
import random as _rnd
_idx = _rnd.sample(range(len(train_ds)), min(5, len(train_ds)))
for i in _idx:
    t = train_ds[i]['text']
    ok_turn = '<start_of_turn>' in t
    ok_sys  = 'medical assistant' in t
    flag = '' if (ok_turn and ok_sys) else '  <-- PERIKSA!'
    print(f'--- #{i} | turn_marker={ok_turn} system={ok_sys}{flag} ---')
    print(t[:400])
    print()
print(f'Train: {len(train_ds):,}  Val: {len(val_ds):,}')

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=LORA_TARGET,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROP,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=SEED,
)
model.print_trainable_parameters()

In [ ]:
from trl import SFTConfig, SFTTrainer
from transformers import EarlyStoppingCallback
from unsloth.chat_templates import train_on_responses_only
import glob

os.makedirs(CKPT_DIR, exist_ok=True)
USE_BF16 = torch.cuda.is_bf16_supported()

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    args=SFTConfig(
        output_dir=CKPT_DIR,
        dataset_text_field='text',
        max_length=MAX_SEQ_LEN,
        packing=False,                 # WAJIB False utk train_on_responses_only (mask prompt)
        dataset_num_proc=2,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_ratio=WARMUP_RATIO,
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        optim='adamw_8bit',
        lr_scheduler_type='cosine',
        fp16=not USE_BF16,
        bf16=USE_BF16,
        max_grad_norm=1.0,
        logging_steps=50,
        eval_strategy='epoch',
        save_strategy='epoch',         # cocok dgn eval -> load_best_model_at_end + EarlyStopping
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model='eval_loss',
        greater_is_better=False,
        report_to='none',
        dataloader_num_workers=0,
        dataloader_pin_memory=False,
        seed=SEED,
    ),
)

# Completion-only: loss HANYA pada jawaban assistant (system+user di-mask).
# Fix utama val_loss tinggi + instruction-following lemah.
# Marker turn Gemma-3: user '<start_of_turn>user\n', model '<start_of_turn>model\n'.
trainer = train_on_responses_only(
    trainer,
    instruction_part='<start_of_turn>user\n',
    response_part='<start_of_turn>model\n',
)

trainer.add_callback(EarlyStoppingCallback(
    early_stopping_patience=2,
    early_stopping_threshold=0.001,
))

# Kuota terbatas -> resume dari checkpoint terakhir di CKPT_DIR (-v2) jika ada.
# Hanya resume dari checkpoint v2 sendiri; folder v1 tidak disentuh.
_ckpts = sorted(glob.glob(os.path.join(CKPT_DIR, 'checkpoint-*')),
                key=lambda p: int(p.rsplit('-', 1)[-1]))
_resume = _ckpts[-1] if _ckpts else None
if _resume:
    print(f'Resuming from checkpoint: {_resume}')

torch.cuda.empty_cache()
try:
    result = trainer.train(resume_from_checkpoint=_resume)
except KeyError as e:
    # Checkpoint lama dibuat tanpa state callback (mis. EarlyStopping) ->
    # 'KeyError: EarlyStoppingCallback'. Aman: mulai training dari awal.
    print(f'[WARN] Gagal resume ({e!r}); mulai training dari awal.')
    result = trainer.train()
print(f'Train loss : {result.training_loss:.4f}  Steps : {result.global_step}')

## Kurva Learning Rate & Loss

In [ ]:
import matplotlib.pyplot as plt

log_hist   = trainer.state.log_history
lr_steps   = [(h['step'], h['learning_rate']) for h in log_hist if 'learning_rate' in h]
loss_steps = [(h['step'], h['loss'])          for h in log_hist if 'loss' in h]
eval_steps = [(h['step'], h['eval_loss'])     for h in log_hist if 'eval_loss' in h]

fig, ax = plt.subplots(1, 2, figsize=(11, 4))

if lr_steps:
    xs, ys = zip(*lr_steps)
    ax[0].plot(xs, ys)
ax[0].set_title('Learning Rate Schedule')
ax[0].set_xlabel('step'); ax[0].set_ylabel('learning rate')

if loss_steps:
    xs, ys = zip(*loss_steps)
    ax[1].plot(xs, ys, label='train_loss')
if eval_steps:
    xs, ys = zip(*eval_steps)
    ax[1].plot(xs, ys, 'o-', label='eval_loss')
ax[1].set_title('Loss')
ax[1].set_xlabel('step'); ax[1].set_ylabel('loss'); ax[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(CKPT_DIR, 'training_curves.png'), dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
os.makedirs(ADAPTER_DIR, exist_ok=True)
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f'Adapter saved -> {ADAPTER_DIR}')

## Evaluasi ROUGE-L

In [ ]:
import warnings
warnings.filterwarnings('ignore', message='.*max_new_tokens.*')

if not os.path.exists(TEST_FILE):
    print(f'[SKIP] test.jsonl tidak ditemukan: {TEST_FILE}')
else:
    from datasets import load_dataset as _lds
    from evaluate import load as load_metric
    from tqdm.auto import tqdm

    rouge = load_metric('rouge')
    FastLanguageModel.for_inference(model)  # Unsloth: 2x lebih cepat untuk inferensi
    torch.cuda.empty_cache()

    preds, refs = [], []
    test_ds = _lds('json', data_files=TEST_FILE, split='train')
    n_eval  = min(200, len(test_ds))
    for sample in tqdm(test_ds.select(range(n_eval)), total=n_eval, desc='ROUGE-L'):
        msgs = sample['messages']
        ref  = next(m['content'] for m in msgs if m['role'] == 'assistant')
        prompt_msgs = to_gemma_messages([m for m in msgs if m['role'] != 'assistant'])
        text   = tokenizer.apply_chat_template(prompt_msgs, tokenize=False,
                                               add_generation_prompt=True)
        inputs = tokenizer(text, return_tensors='pt', truncation=True,
                           max_length=MAX_SEQ_LEN - 256).to('cuda')
        with torch.inference_mode():
            out = model.generate(**inputs, max_new_tokens=256, do_sample=False,
                                 no_repeat_ngram_size=3,
                                 pad_token_id=tokenizer.eos_token_id)
        preds.append(tokenizer.decode(out[0][inputs['input_ids'].shape[1]:],
                                      skip_special_tokens=True))
        refs.append(ref)
        del inputs, out
        torch.cuda.empty_cache()

    scores = rouge.compute(predictions=preds, references=refs, use_stemmer=True)
    for k, v in scores.items():
        print(f'  {k}: {v:.4f}')

## Uji inferensi kualitatif

In [ ]:
test_questions = [
    'Apa gejala utama demam berdarah dengue dan kapan harus ke rumah sakit?',
    'Bagaimana cara menangani pasien dengan nyeri dada akut?',
    'What are the first-line treatment options for Type 2 Diabetes Mellitus?',
    'Apakah antibiotik diperlukan untuk infeksi virus seperti flu biasa?',
    'Dok saya hamil 8 minggu dan mengalami mual parah, apa yang harus saya lakukan?',
]
FastLanguageModel.for_inference(model)  # Unsloth: 2x lebih cepat untuk inferensi
for q in test_questions:
    messages = to_gemma_messages([{'role': 'user', 'content': q}])  # system terpadu di-inject
    text   = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to('cuda')
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=256, do_sample=True,
                             temperature=0.6, top_p=0.9, repetition_penalty=1.15,
                             no_repeat_ngram_size=3,
                             pad_token_id=tokenizer.eos_token_id)
    resp = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    print(f'Q: {q}\nA: {resp[:400]}\n' + '-' * 70)
    del inputs, out
    torch.cuda.empty_cache()

## Merge adapter -> model penuh

In [ ]:
if DO_MERGE:
    os.makedirs(MERGED_DIR, exist_ok=True)
    model.save_pretrained_merged(MERGED_DIR, tokenizer, save_method='merged_16bit')
    print(f'Merged saved -> {MERGED_DIR}')
else:
    print('DO_MERGE=False. Adapter tersimpan di', ADAPTER_DIR)

In [ ]:
if PUSH_TO_HUB:
    if not HF_TOKEN:
        raise ValueError('Set HF_TOKEN di Colab Secrets untuk push ke HuggingFace Hub')
    model.push_to_hub_merged(HF_REPO, tokenizer, save_method='merged_16bit', token=HF_TOKEN)
    print(f'Model pushed -> huggingface.co/{HF_REPO}')
else:
    print(f'Output lokal:\n  Adapter: {ADAPTER_DIR}')
    if DO_MERGE:
        print(f'  Merged : {MERGED_DIR}')